# Agent 1 — Mandate

**What this notebook does:**  
Defines and records the investment mandate — the rules and objectives that govern every decision made downstream. Every portfolio choice, every exclusion, every ESG weight traces back to this document.

**How to present this to investors:**  
> *Before any data is processed, our mandate agent encodes the fund's investment philosophy, constraints, and ESG priorities into a machine-readable form. This ensures the pipeline is rule-governed rather than discretionary — every holding can be justified by reference to a specific mandate clause.*

## Step 1 — Define the mandate

Edit the fields below to reflect your team's agreed investment thesis.  
This cell is the **single source of truth** — change it here and all downstream notebooks inherit the updated rules.

In [1]:
import json
import os
from datetime import date

TODAY = str(date.today())

# ============================================================
#  MANDATE — edit these values to match your team's decisions
# ============================================================

MANDATE = {
    # --- Fund identity
    "fund_name": "ESADE Sustainable European Equity Fund",
    "vintage": TODAY,
    "universe": "STOXX Europe 600 subset (50+ companies analysed)",
    "benchmark": "STOXX Europe 600",

    # --- Investment thesis (3 sentences — memorise this for Q&A)
    "investment_thesis": (
        "We construct a concentrated long-only European equity portfolio "
        "of 15–25 holdings by integrating fundamental financial quality with "
        "best-in-class ESG credentials, verified against EU Taxonomy alignment "
        "and independently screened for greenwashing. "
        "We believe companies with credible sustainability commitments carry "
        "lower regulatory, reputational, and transition risk over a 5-year horizon. "
        "All exclusions and overrides are documented, creating an auditable process "
        "that can withstand scrutiny under SFDR Article 8 principles."
    ),

    # --- Portfolio constraints
    "constraints": {
        "min_holdings": 15,
        "max_holdings": 25,
        "target_holdings": 20,
        "max_single_weight_pct": 10.0,   # no stock above 10%
        "min_sectors": 5,
        "strategy": "long-only",
        "currency": "EUR"
    },

    # --- Scoring weights (must sum to 1.0)
    "composite_score_weights": {
        "esg": 0.60,         # ESG composite (E+S+G)
        "financial": 0.40    # Sharpe / quality metrics
    },
    "esg_pillar_weights": {
        "environmental": 0.40,
        "social": 0.30,
        "governance": 0.30
    },

    # --- Hard exclusions (any company meeting these criteria is removed before scoring)
    "hard_exclusions": [
        "Thermal coal revenue > 5%",
        "Tobacco production",
        "Controversial weapons (cluster munitions, antipersonnel mines)",
        "Greenwashing 8-Test score HIGH on >= 3 dimensions"
    ],

    # --- Watchlist triggers (company is flagged but not automatically excluded)
    "watchlist_triggers": [
        "Greenwashing 8-Test score HIGH on 2 dimensions",
        "SBTi target not yet approved",
        "Scope 3 emissions not disclosed",
        "Board gender diversity below 30%"
    ],

    # --- Required metrics in final portfolio
    "required_metrics": [
        "WACI (Weighted Average Carbon Intensity)",
        "ESG composite score (E, S, G, aggregate)",
        "Biodiversity / nature-risk proxy",
        "EU Taxonomy eligibility",
        "Greenwashing 8-Test result per holding"
    ],

    # --- Human override policy
    "human_override_policy": (
        "Team may override quantitative ranking for any holding. "
        "Every override must be logged in notebook 10_human_review.ipynb "
        "with a written rationale and the team member responsible."
    )
}

# Validate weights sum to 1.0
composite_sum = sum(MANDATE["composite_score_weights"].values())
esg_sum = sum(MANDATE["esg_pillar_weights"].values())
assert abs(composite_sum - 1.0) < 0.001, f"Composite weights sum to {composite_sum}, not 1.0"
assert abs(esg_sum - 1.0) < 0.001, f"ESG pillar weights sum to {esg_sum}, not 1.0"

print("Mandate defined successfully.")
print(f"Fund: {MANDATE['fund_name']}")
print(f"Target holdings: {MANDATE['constraints']['target_holdings']}")
print(f"ESG weight: {MANDATE['composite_score_weights']['esg']*100:.0f}% | Financial weight: {MANDATE['composite_score_weights']['financial']*100:.0f}%")
print(f"Hard exclusions: {len(MANDATE['hard_exclusions'])} rules")

Mandate defined successfully.
Fund: ESADE Sustainable European Equity Fund
Target holdings: 20
ESG weight: 60% | Financial weight: 40%
Hard exclusions: 4 rules


## Step 2 — Print the one-page mandate summary

This is the text you paste into your report Section 1 and into the first slide of your deck.

In [2]:
print("=" * 65)
print(f"  {MANDATE['fund_name'].upper()}")
print(f"  Investment Mandate — as of {MANDATE['vintage']}")
print("=" * 65)

print("\nINVESTMENT THESIS")
print("-" * 65)
# Word-wrap the thesis at ~65 characters
import textwrap
for line in textwrap.wrap(MANDATE['investment_thesis'], 65):
    print(line)

print("\nPORTFOLIO CONSTRAINTS")
print("-" * 65)
c = MANDATE['constraints']
print(f"  Holdings:   {c['min_holdings']}–{c['max_holdings']} stocks (target {c['target_holdings']})")
print(f"  Max weight: {c['max_single_weight_pct']}% per holding")
print(f"  Min sectors: {c['min_sectors']}")
print(f"  Strategy:   {c['strategy'].capitalize()}, {c['currency']}")
print(f"  Benchmark:  {MANDATE['benchmark']}")

print("\nSCORING WEIGHTS")
print("-" * 65)
w = MANDATE['composite_score_weights']
p = MANDATE['esg_pillar_weights']
print(f"  Composite:  ESG {w['esg']*100:.0f}% | Financial {w['financial']*100:.0f}%")
print(f"  ESG pillars: E {p['environmental']*100:.0f}% | S {p['social']*100:.0f}% | G {p['governance']*100:.0f}%")

print("\nHARD EXCLUSIONS")
print("-" * 65)
for rule in MANDATE['hard_exclusions']:
    print(f"  ✗ {rule}")

print("\nWATCHLIST TRIGGERS")
print("-" * 65)
for rule in MANDATE['watchlist_triggers']:
    print(f"  ⚠ {rule}")

print("=" * 65)

  ESADE SUSTAINABLE EUROPEAN EQUITY FUND
  Investment Mandate — as of 2026-05-06

INVESTMENT THESIS
-----------------------------------------------------------------
We construct a concentrated long-only European equity portfolio
of 15–25 holdings by integrating fundamental financial quality
with best-in-class ESG credentials, verified against EU Taxonomy
alignment and independently screened for greenwashing. We believe
companies with credible sustainability commitments carry lower
regulatory, reputational, and transition risk over a 5-year
horizon. All exclusions and overrides are documented, creating an
auditable process that can withstand scrutiny under SFDR Article
8 principles.

PORTFOLIO CONSTRAINTS
-----------------------------------------------------------------
  Holdings:   15–25 stocks (target 20)
  Max weight: 10.0% per holding
  Min sectors: 5
  Strategy:   Long-only, EUR
  Benchmark:  STOXX Europe 600

SCORING WEIGHTS
------------------------------------------------------

## Step 3 — Save the mandate as a machine-readable file

Downstream notebooks read this file to apply exclusion rules and scoring weights automatically.

In [3]:
os.makedirs("../outputs/scores", exist_ok=True)
mandate_path = "../outputs/scores/mandate.json"

with open(mandate_path, "w") as f:
    json.dump(MANDATE, f, indent=2)

print(f"Mandate saved to: {mandate_path}")
print("All downstream notebooks will read this file for rules and weights.")

Mandate saved to: ../outputs/scores/mandate.json
All downstream notebooks will read this file for rules and weights.


## ✅ Notebook complete

You now have:
- A **written investment thesis** ready to paste into your report
- A **machine-readable mandate** (`outputs/scores/mandate.json`) consumed by all other notebooks
- **Scoring weights** and **exclusion rules** set in one place

**Next:** Open `01_data_ingestion.ipynb` to load the four CSV files and download market prices.